# Step 2 — What does each column mean, and which values does it hold?

**Purpose.** Before asking questions we need to know every column: what it means and which values it really contains. This notebook prints, for each of the three raw tables, the data type, the number of missing and distinct values, and example values. For columns with few distinct values it prints all of them with their counts.

**Where the meanings come from.** The author only defines some columns. Every meaning below carries a tag:

| Tag | Meaning |
|---|---|
| **[A]** | The author states it in the Kaggle description or column descriptions |
| **[I]** | Inferred from the column name or from sample values, **not confirmed**. Check it against the output below |
| **[V]** | Verified by us from the real output of this notebook |

After running the notebook, change an **[I]** to **[V]** only when the printed values support it, and fix the meaning when they don't.

**How to run on Kaggle.** Add the *MyAnimeList Dataset* as input and run all cells. Parts 1 and 2 take seconds. Part 3 reads the 80M-row list table once and takes a few minutes; set `RUN_LISTS = False` in the setup cell to skip the full read.

**Where this step sits.** Step 2 of the exploration order (see [docs/exploration_order.md](../docs/exploration_order.md)). It only looks at values and formats. The tags [V] that come from the statistical tests of step 5 were obtained on raw data and are re-confirmed after cleaning. Part 4 answers three questions that parts 1 to 3 left open.


## Data dictionary

### Anime table — `AnimeList.csv`, one row is an anime (31 columns)

| Column | Meaning | Tag |
|---|---|---|
| `anime_id` | MyAnimeList id of the anime, the key of this table | [A] |
| `title` | Title (romanized) | [A] |
| `title_english` | English title | [A] |
| `title_japanese` | Japanese title | [A] |
| `title_synonyms` | Other names of the anime | [A] |
| `image_url` | URL of the cover image | [A] |
| `type` | Format of the anime: Movie, Music, ONA, OVA, Special, TV, or the text `Unknown` (30 anime) | [V] |
| `source` | What the anime is based on, 16 values such as Manga, Original, Light novel; `Unknown` is a real value for 4,210 anime (29%) | [V] |
| `episodes` | Number of episodes; 0 means unknown (all 30 `Unknown` type, 272 TV and 118 ONA anime have 0). Movies have a median of 1 | [V] |
| `status` | Airing status: Finished Airing (13,791), Currently Airing (351), Not yet aired (336). The 336 is the number of anime **labelled** not yet aired. Tested against start dates and list rows (part 1 and part 3): the label is believable for almost all of them (6 have completed rows, and 2 of them have real start dates on 13 and 18 May 2018) | [V] |
| `airing` | True for 351 anime. Checked row by row: identical to `status = Currently Airing` (0 disagreements), so it is redundant with `status` | [V] |
| `aired_string` | Air dates as text, for example `Jan 13, 2012 to Mar 30, 2012` | [V] |
| `aired` | Air dates as the text of a dictionary, for example `{'from': '2012-01-13', 'to': '2012-03-30'}`. A date is real only when `aired_string` writes a full date. For a partial text (a month and a year, or a year only) the start is empty (1,468 anime) or made up (227 start dates: the month May for a year-only text, the days 9 to 12, mostly the year 2018; part 4) | [V] |
| `duration` | Length of one episode as text, for example `24 min. per ep.` (301 distinct texts) | [V] |
| `rating` | Audience rating: a content label that implies a recommended age. It is not the users' score. 7 values: G, PG, PG-13, R, R+, Rx and the text `None` (544 anime) | [V] |
| `score` | Average of the users' scores (correlation 0.976 with the mean of our users' scores, ours are 0.17 lower on average) | [V] |
| `scored_by` | Number of users who scored it (rank correlation 0.995 with our own count). For 59 anime our count is larger than this value (up to 64 times), so use it for the order of anime, not as an exact count | [V] |
| `rank` | Rank by score, stored as a decimal number but holds integers only; 10.9% missing. Rank correlation with `score` is only -0.855, and -0.862 with `members`, so it is not simply the order of `score` | [A] |
| `popularity` | Rank by number of members, 1 = most popular (rank correlation -0.996 with `members`, -0.737 with `score`). 9 anime have 0 or less and values repeat (11,759 distinct for 14,478 anime) | [V] |
| `members` | Number of users who have the anime on their list (rank correlation 0.994 with our own count; our count is larger for only 2 anime, at most 18% above) | [V] |
| `favorites` | Number of users who marked it as favorite. Cannot be recomputed from our tables; larger than `members` for 1 anime | [I] |
| `background` | Free-text background information; 92.7% missing | [V] |
| `premiered` | Season it premiered in, for example `Winter 2012`; 71.7% missing | [V] |
| `broadcast` | Weekday and time, for example `Tuesdays at 22:00 (JST)`; contains placeholder texts such as `Unknown`; 70.5% missing | [V] |
| `related` | Related anime as the text of a dictionary: relation type, then a list of items with `mal_id`, `type` (anime or manga), `url` and `title`. 4,515 anime (31.2%) have no relation, written as an empty list `[]` | [V] |
| `producer` | Producers separated by commas; 42.8% missing | [V] |
| `licensor` | Licensors separated by commas; 76.7% missing | [V] |
| `studio` | Animation studio(s) separated by commas; 41.0% missing | [V] |
| `genre` | Genres separated by commas; 0.4% missing | [V] |
| `opening_theme` | Opening songs stored as the text of a list, for example `['"Nirvana" by MUCC']` | [V] |
| `ending_theme` | Ending songs stored as the text of a list | [V] |

### User table — `UserList.csv`, one row is a user (17 columns)

| Column | Meaning | Tag |
|---|---|---|
| `username` | The user's name, used as the user key | [A] |
| `user_id` | Numeric id of the user. **Not unique:** 302,673 distinct values for 302,675 rows, because 2 ids are each used by 2 usernames. Each pair is one account crawled under two names, and the same list is stored under both names (part 4) | [I] |
| `user_watching` | Number of anime on the user's list with status watching | [I] |
| `user_completed` | ... with status completed | [I] |
| `user_onhold` | ... with status on hold | [I] |
| `user_dropped` | ... with status dropped | [I] |
| `user_plantowatch` | ... with status plan to watch | [I] |
| `user_days_spent_watching` | Total days the user spent watching | [I] |
| `gender` | Gender: Female, Male, Non-Binary; 28.0% missing | [V] |
| `location` | Free text typed by the user (country, state or city, inconsistent); 48.2% missing, 55,283 distinct values | [V] |
| `birth_date` | Birth date as `YYYY-MM-DD`; 44.3% missing | [V] |
| `access_rank` | Empty in all 302,675 rows, so it carries no information | [V] |
| `join_date` | Registration date as `YYYY-MM-DD`; 129 missing | [V] |
| `last_online` | Last time online with time of day; contains values from the year 1900; 129 missing | [V] |
| `stats_mean_score` | The user's mean score, 2 decimals; within 0.05 of the mean of the user's own scores for 99.7% of users | [V] |
| `stats_rewatched` | A total count of rewatching. Larger than the user's list size for 5,086 users, so it is not the number of anime rewatched; whether it counts times or episodes is unknown | [I] |
| `stats_episodes` | Total episodes watched; median ratio 1.000 with our sum of `my_watched_episodes`, equal for 50.8% of users; has impossible values such as 10,989,789 | [V] |

The five `user_*` count columns are described by the author only as "aggregated values from their anime lists".

### List table — `UserAnimeList.csv`, one row is a user–anime pair (11 columns)

| Column | Meaning | Tag |
|---|---|---|
| `username` | The user | [A] |
| `anime_id` | The anime | [A] |
| `my_watched_episodes` | Episodes the user has watched | [I] |
| `my_start_date` | Date the user started; 86.2% are `0000-00-00`, and some are partial such as `2008-00-00` | [V] |
| `my_finish_date` | Date the user finished; 83.1% are `0000-00-00`, and some are partial such as `2012-08-00` | [V] |
| `my_score` | Score the user gave, 1–10. **0 means not rated**, since the 1–10 scale has no 0 | [A] |
| `my_status` | Status code: 1 watching, 2 completed, 3 on hold, 4 dropped, 6 plan to watch. **The file stores numbers, the words come from the description.** Other codes are unexplained | [A] |
| `my_rewatching` | 0 or 1, or empty. 63,288,415 rows are 0, 45,850 are 1, 16,741,847 are empty | [V] |
| `my_rewatching_ep` | An episode number for a rewatch of that anime: only 11,964 rows (0.015%) are above 0, median 10, at most the anime's episode count in 97.8% of compared rows, maximum 16,777,215 (about 35 rows). Mostly on rows where `my_rewatching` is 0 | [I] |
| `my_last_updated` | Unix timestamp in seconds, for example 1362307973 is 2013-03-03 10:52:53 UTC | [V] |
| `my_tags` | Free-text tags written by the user; 91.3% missing in the first million rows | [V] |


In [1]:
from pathlib import Path
import sys

import pandas as pd

# Works both on Kaggle and inside the repo (with the data placed in data/raw/)
KAGGLE_INPUT = Path("/kaggle/input")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "environment.yml").exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
elif ROOT is not None:
    DATA_DIR = ROOT / "data" / "raw"
else:
    raise RuntimeError("Data not found: run on Kaggle or inside the repository.")

paths = {p.name: p for p in DATA_DIR.rglob("*.csv")}
RUN_LISTS = True  # False skips the full read of the 80M-row list table
na_kwargs = dict(keep_default_na=False, na_values=[""])  # only an empty cell is missing

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 200)

print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)
print("Number of CSV files:", len(paths))


def profile(df, n_examples=4):
    """One row per column: type, missing, distinct values and a few example values."""
    rows = []
    for col in df.columns:
        s = df[col]
        rows.append({
            "column": col,
            "dtype": str(s.dtype),
            "missing": int(s.isna().sum()),
            "missing_pct": round(s.isna().mean() * 100, 2),
            "distinct": int(s.nunique(dropna=True)),
            "examples": s.dropna().astype(str).str.slice(0, 45).unique()[:n_examples].tolist(),
        })
    return pd.DataFrame(rows).set_index("column")


def show_values(df, max_distinct=25):
    """Print every distinct value, with its count, for columns that have few distinct values."""
    for col in df.columns:
        distinct = df[col].nunique(dropna=False)
        if distinct <= max_distinct:
            counts = df[col].value_counts(dropna=False)
            try:
                counts = counts.sort_index()
            except TypeError:
                pass
            print(f"\n--- {col}: {distinct} distinct values")
            print(counts.to_string())

Python: 3.13.18 | pandas: 2.3.3
Number of CSV files: 9


## 1. Anime table

For every column: its type, how many values are missing, how many distinct values it has, and a few examples. Read the `examples` column slowly and compare it with the dictionary above.

In [2]:
anime = pd.read_csv(paths["AnimeList.csv"], **na_kwargs)
print("shape:", anime.shape, "| anime_id unique:", anime.anime_id.is_unique)
print(profile(anime).to_string())

shape: (14478, 31) | anime_id unique: True
                  dtype  missing  missing_pct  distinct                                                                                                                                                                                      examples
column                                                                                                                                                                                                                                               
anime_id          int64        0         0.00     14478                                                                                                                                                                      [11013, 2104, 5262, 721]
title            object        0         0.00     14477                                                                                                                          [Inu x Boku SS, Seto no Hanayome, Shugo Ch

In [3]:
# Every distinct value of the columns that have few of them
show_values(anime)


--- type: 7 distinct values
type
Movie      2624
Music       849
ONA        1144
OVA        3612
Special    1948
TV         4271
Unknown      30

--- source: 16 distinct values
source
4-koma manga      224
Book               93
Card game          59
Digital manga      10
Game              607
Light novel       563
Manga            3120
Music             314
Novel             359
Original         3368
Other             408
Picture book       96
Radio               9
Unknown          4210
Visual novel      888
Web manga         150

--- status: 3 distinct values
status
Currently Airing      351
Finished Airing     13791
Not yet aired         336

--- airing: 2 distinct values
airing
False    14127
True       351

--- rating: 7 distinct values
rating
G - All Ages                      4541
None                               544
PG - Children                     1279
PG-13 - Teens 13 or older         5020
R - 17+ (violence & profanity)     997
R+ - Mild Nudity                   878
Rx - He

### NEW — `airing` against `status`, row by row (added after the first run)

**Why this check exists.** The dictionary says `airing` looks redundant with `status`. The first run only compared two **counts**: 351 anime have `airing = True` and 351 anime have `status = Currently Airing`. Equal counts do not prove that the two columns describe the same anime. They could be 351 different anime in each column.

**What the cell does.** It counts every combination of the two columns, row by row, and counts the anime where they disagree.

**How to read the result.**
- All 351 anime sit in the cell (`Currently Airing`, `True`) and the disagreement count is **0**: the two columns say the same thing on every anime, so `airing` is redundant and we can drop it later.
- Any other number: the columns disagree on some anime. We look at those anime before deciding, and the dictionary row of `airing` stays [I].


In [4]:
# NEW: row-by-row check of airing against status (the first run compared only the two counts)
print(pd.crosstab(anime["status"], anime["airing"], margins=True))
disagree = anime["status"].eq("Currently Airing") != anime["airing"]
print("\nanime where airing disagrees with status = Currently Airing:", int(disagree.sum()))


airing            False  True    All
status                              
Currently Airing      0   351    351
Finished Airing   13791     0  13791
Not yet aired       336     0    336
All               14127   351  14478

anime where airing disagrees with status = Currently Airing: 0


**Result: `airing` is redundant with `status`.** The table has no cell outside the diagonal. All 351 `Currently Airing` anime have `airing = True`, and all 13,791 `Finished Airing` and 336 `Not yet aired` anime have `airing = False`. The count of disagreements is **0**.

So `airing` holds no information that `status` does not already hold, and in the cleaning step we can keep one of the two. The first run only compared two counts (351 and 351). Now every row has been compared, so the tag of `airing` is back to [V].


### NEW — is the label "Not yet aired" believable? (added after the first run)

**Why this check exists.** `status = Not yet aired` is a label that MyAnimeList sets by hand and the crawl took months, so some of these anime may already have been airing. The label is relative to the crawl date (around 2018), not to today. We cannot say "336 anime have not aired", only "336 anime are **labelled** not yet aired". Other columns give independent evidence:

| Evidence | If the label is right | If the label is wrong |
|---|---|---|
| Start date inside `aired`, against a crawl date | Start date is later than the crawl date, or missing | Start date is already in the past |
| `scored_by` and `members` | Zero or very small | Large numbers |
| Rows of the list table with `my_status = 2` (completed), part 3 below | None or almost none | Many users "completed" it |

**The crawl date.** We do not know it exactly. A lower bound is the latest start date among the anime labelled `Currently Airing`: they were airing when the page was read, so the crawl cannot be earlier than their start date.

**How to read the result.** The count "start date on or before that bound" is the number of anime whose label looks stale. The ten earliest start dates are printed so we can look at the worst cases.


In [5]:
# NEW: is the label "Not yet aired" believable? Evidence from other columns
import ast

def start_date(text):
    # `aired` is the text of a Python dictionary such as {'from': '2012-01-13T00:00:00+00:00', 'to': None}
    try:
        return pd.to_datetime(ast.literal_eval(text).get("from"), utc=True, errors="coerce")
    except Exception:
        return pd.NaT

anime["start"] = anime["aired"].map(start_date)
nya = anime[anime["status"] == "Not yet aired"]
bound = anime.loc[anime["status"] == "Currently Airing", "start"].max()

print("latest start date among Currently Airing anime (lower bound of the crawl date):", bound)
print("labelled Not yet aired:", len(nya))
print("  start date missing:", int(nya["start"].isna().sum()))
print("  start date later than the bound:", int((nya["start"] > bound).sum()))
print("  start date on or before the bound (label looks stale):", int((nya["start"] <= bound).sum()))
print("  scored_by > 0:", int((nya["scored_by"] > 0).sum()), "| members > 0:", int((nya["members"] > 0).sum()))
print()
print("the ten earliest start dates among them:")
print(nya.sort_values("start")[["anime_id", "title", "type", "start", "scored_by", "members"]].head(10).to_string(index=False))


latest start date among Currently Airing anime (lower bound of the crawl date): 2018-05-25 00:00:00+00:00
labelled Not yet aired: 336
  start date missing: 147
  start date later than the bound: 129
  start date on or before the bound (label looks stale): 60
  scored_by > 0: 46 | members > 0: 332

the ten earliest start dates among them:
 anime_id                                   title  type                     start  scored_by  members
    37436                               Revisions    TV 2018-01-11 00:00:00+00:00          0     1477
    37435                         Carol & Tuesday    TV 2018-04-10 00:00:00+00:00          0     1180
    32843             Senki Zesshou Symphogear XV    TV 2018-04-10 00:00:00+00:00          0     4738
    33735                             Locker Room   ONA 2018-04-11 00:00:00+00:00          0      615
    36792                     Eromanga-sensei OVA   OVA 2018-05-09 00:00:00+00:00          0    18957
    35960                              Kud Wafte

**Result (start dates and ratings). The label is consistent with the data; most of the early start dates are not real dates.**

| What we counted among the 336 anime labelled `Not yet aired` | Anime | Share |
|---|---|---|
| Start date missing | 147 | 43.8% |
| Start date later than the crawl bound (2018-05-25) | 129 | 38.4% |
| Start date on or before the bound (the label looks stale on this test alone) | 60 | 17.9% |
| `scored_by` above 0 | 46 | 13.7% |
| `members` above 0 | 332 | 98.8% |

- **276 of 336 (82.1%) have no start date or a date after the bound.** This is what we expect from anime that have not aired.
- **60 anime have a start date on or before the bound, but for 54 of them that date is not a real start date.** Part 4 compares `aired` with the text `aired_string`: 49 of the 60 are written only as a year (`2018 to ?`, `2019 to ?`) and 5 only as a month and a year (`Apr, 2019 to ?`), yet `aired` holds a complete date for them, and the same day numbers (9 to 12) and the month May come back again and again. Only 6 have a full date in the text (13 to 25 May 2018), so this test says nothing about the other 54. The ten earliest dates in the output belong to anime with `scored_by = 0` (nobody has rated them), while `members` goes from 331 to 18,957, which agrees with anime that have not aired. Part 3 gives the stronger test, and part 4 followed it up: 58 of the 60 early anime behave like the other anime that have not aired (almost only plan-to-watch rows, few raters), and the 2 that look different have real start dates (13 and 18 May 2018), which part 4 explains.
- **46 anime have `scored_by` above 0, so unaired anime can have scores.** This is consistent with notebook 01 (2.2 a), where 238,166 plan-to-watch rows carry a score: a user can score an anime before watching it.
- **332 of 336 have members.** People add an anime to their plans before it airs, so members above 0 is normal.
- **The crawl date.** The bound from the anime table (2018-05-25) is a few days later than the latest `my_last_updated` of the list table (2018-05-22, see part 3). The two views of the crawl date are within a few days of each other, around the end of May 2018. Part 4 shows that some start dates of this table were made up, with the days 9 to 12 May (and 25 May for three anime); whether the date that defines the bound is real was not checked.


### Observations for part 1 (anime table)

**Rows and key.** The table has 14,478 rows and `anime_id` is unique, so it identifies each anime. `title` has 14,477 distinct values because two anime share the title `Hinamatsuri`. In `image_url` the examples look identical because the profile function cuts every example at 45 characters. The column has 96 missing values and 14,382 distinct ones, so every non-missing URL is different.

**Which meanings are confirmed.** A meaning is **confirmed [V]** when the real values behave the way the meaning predicts. It stays **inferred [I]** when the values only look plausible. After this stage, `type`, `source`, `status`, `airing`, `aired_string`, `aired`, `duration`, `rating`, `premiered`, `broadcast`, `related`, `producer`, `licensor`, `studio`, `genre`, `opening_theme`, `ending_theme` and `background` were [V]. `episodes`, `score`, `scored_by`, `popularity`, `members` and `favorites` were still [I], because numbers can look plausible without proving what they count. Step 5a tests them with code.

**Placeholders written as words.** Some cells hold a word that means "no value". They are real text, so `isna()` does not see them as missing:
- `type = Unknown` (30 anime), `source = Unknown` (4,210 anime, 29%), `rating = None` (544 anime), and texts like `Unknown` or `Fridays at Unknown` inside `broadcast`.
- Pandas would normally turn the text `None` into a missing value. We read the file with `keep_default_na=False`, so `rating = None` stayed visible. Whether to count these words as missing is a cleaning decision for the team.

**Anime labelled Not yet aired.** 336 anime carry this label. We tested whether it is believable with three pieces of evidence, and the results are right under the check cells. The list table says yes for almost all of them (99.8% of the 543,024 rows of these anime are plans, and only 6 anime have a completed row). The start dates alone suggest 60 stale labels, but 54 of those dates are made up from a partial text (part 4). Proposal for step 4: keep `status` as given, leave these anime out of score questions, and flag the 6 with completed rows (2 of them started on 13 and 18 May 2018, see part 4). `airing` is redundant with `status` (0 disagreements out of 14,478).

**Missing data.** *Missing* here means an empty cell.

| Level | Columns | Meaning for us |
|---|---|---|
| Above 70% missing | `background` 92.7%, `licensor` 76.7%, `premiered` 71.7%, `broadcast` 70.5% | Not usable as features |
| 38–61% missing | `title_english` 60.5%, `producer` 42.8%, `studio` 41.0%, `title_synonyms` 38.3% | Usable only for the anime that have them |
| About 11% | `rank` (1,574 missing) | Anime without a rank |
| Under 1% | `genre` 0.4%, `title_japanese` 0.2%, `image_url` 0.7% | Fine |

**Formats that need parsing before any analysis.** Parsing means turning a text into a structure we can count.
- `aired` and `related` are the text of Python dictionaries.
- `opening_theme` and `ending_theme` are the text of Python lists.
- `genre`, `producer`, `studio` and `licensor` are comma-separated lists inside one cell.
- `duration` and `aired_string` are free text.

All of these are the "multiple values in one cell" problem from the tidy-data lesson: one cell holds several values, so we must split them into one row per value.

## 2. User table

In [6]:
users = pd.read_csv(paths["UserList.csv"], **na_kwargs)
print("shape:", users.shape, "| username unique:", users.username.is_unique)
print(profile(users).to_string())

shape: (302675, 17) | username unique: True
                            dtype  missing  missing_pct  distinct                                                                              examples
column                                                                                                                                                 
username                   object        0         0.00    302675                                        [karthiga, RedvelvetDaisuki, Damonashu, bskai]
user_id                     int64        0         0.00    302673                                                     [2255153, 1897606, 37326, 228342]
user_watching               int64        0         0.00       580                                                                       [3, 61, 45, 25]
user_completed              int64        0         0.00      2208                                                                   [49, 396, 195, 414]
user_onhold                 int64        0  

In [7]:
show_values(users)

print("\n--- location: top 15 of", users["location"].nunique(), "distinct values")
print(users["location"].value_counts().head(15).to_string())


--- gender: 4 distinct values
gender
Female         81374
Male          135181
Non-Binary      1245
NaN            84875

--- access_rank: 1 distinct values
access_rank
NaN    302675

--- location: top 15 of 55283 distinct values
location
Poland         2425
Germany        1947
Canada         1821
California     1510
Brazil         1459
France         1135
Philippines    1077
Australia      1032
Sweden         1013
Finland         898
Portugal        896
England         874
Singapore       863
USA             709
Texas           694


### NEW — the rows that share a `user_id` (added after the first run)

**Why this check exists.** `username` is unique (302,675 different values) but `user_id` has only 302,673 different values, so some rows share an id with another row. We never looked at those rows. Two explanations are possible, and the output tells them apart:

- **The same account twice.** Everything except the username is equal (join date, last online, statistics, location), as if one account had been crawled under two names. Then one of the two rows is a duplicate.
- **Two different users with a wrong id.** Many columns differ. Then they are two real users and the id is simply not reliable.

**What the cell does.** It lists the shared rows with all their columns (one column per row, so nothing is cut), counts the rows that are identical in every column, and for each shared `user_id` prints the columns in which its rows differ.

**How to read the result.** If the rows differ only in `username`, they look like a duplicated account. If they differ in many columns, they are different users. Either way we keep joining on `username`, never on `user_id`.


In [8]:
# NEW: the rows that share a user_id
dup_ids = users.loc[users["user_id"].duplicated(keep=False), "user_id"].unique()
shared = users[users["user_id"].isin(dup_ids)].sort_values("user_id")
print("user_id values used by more than one row:", len(dup_ids), "| rows involved:", len(shared))
print("rows identical in every column to another row:", int(users.duplicated(keep=False).sum()))
print()
print(shared.T.to_string())
print()
for uid, group in shared.groupby("user_id"):
    differ = [c for c in group.columns if group[c].nunique(dropna=False) > 1]
    print("user_id", uid, "-> the rows differ in:", differ)


user_id values used by more than one row: 2 | rows involved: 4
rows identical in every column to another row: 0

                               1334                 256501   20523                301571
username                  Io_Otonashi            Edengarde   Dojin_                DXJIN
user_id                       3303323              3303323  4053497              4053497
user_watching                      18                   18       28                   28
user_completed                    417                  417      348                  348
user_onhold                         2                    2       14                   14
user_dropped                       42                   42        6                    6
user_plantowatch                   43                   43      141                  141
user_days_spent_watching       112.02               112.02    59.63                59.63
gender                            NaN                 Male      NaN                 Ma

**Result: two ids are each used by two usernames (4 rows). Each pair looks like one account under two names.**

302,675 rows minus 302,673 distinct ids means two extra rows, and they come from **two pairs** of usernames.

| `user_id` | Username with an empty profile | Username with a full profile | List sizes in both rows (watching / completed / on hold / dropped / plan to watch) | Days spent watching |
|---|---|---|---|---|
| 3303323 | `Io_Otonashi` | `Edengarde` | 18 / 417 / 2 / 42 / 43 | 112.02 |
| 4053497 | `Dojin_` | `DXJIN` | 28 / 348 / 14 / 6 / 141 | 59.63 |

- **No row is an exact duplicate of another row** (0 rows identical in every column), because the usernames differ.
- **In each pair the five list-size columns and `user_days_spent_watching` are identical.** The profile columns (`gender`, `location`, `birth_date`, `join_date`, `last_online`, `stats_mean_score`, `stats_rewatched`, `stats_episodes`) are empty in one row and filled in the other.
- **Our reading (confirmed in part 4):** each pair is one MAL account that was crawled under two usernames, for example after a rename. Two different people would not share an id and identical list sizes down to the last anime. The rows with an empty profile are among the 129 users with a missing `join_date` (confirmed in part 4).
- **Part 4 checked the list table: both names own the identical list** (522 rows for each name of the first pair, 537 for the second, same anime, status and score). So 1,059 list rows are duplicated. We still do not know which username is the current one.
- **The effect is tiny** (4 rows of 302,675), and joining on `username` stays correct. `user_id` still cannot be a key.


### Observations for part 2 (user table)

**The `null` fix worked.** When only the empty cell counts as missing, `UserList` has **302,675 rows and 302,675 distinct usernames**, exactly what the author's description says. In the first run of step 1 (notebook 01, read with the pandas defaults) the one missing username was this user named `null`; step 1 was later redone with the safe load.

**`user_id` is not unique.** There are 302,673 distinct `user_id` values for 302,675 rows, because **2 ids are each used by 2 usernames** (4 rows, see the result below the check). Each pair is one account under two names, with the same list under both (part 4). A column that is not unique cannot be a key, so `username` is the safe key for joining tables.

**Gender.** The labels are clean (no mix of `male`, `M` and so on): Male 135,181, Female 81,374, Non-Binary 1,245. But **28.0% of the users (84,875) have no gender**. Any gender comparison therefore covers only 72% of the users, and the users who left it empty may be different from the rest.

**Location is free text.** 48.2% are missing, and the rest has 55,283 distinct values, because every user typed their own text. The values mix levels (Poland and Germany, but also California and Texas as separate entries next to USA) and have formatting differences such as `Chennai, India ` with a space at the end. Any analysis by country needs a grouping step first.

**Dates and ages.** `birth_date` is missing for 44.3% of the users, so an age analysis uses only 55.8% of them. `join_date` holds a date only. `last_online` also holds a time of day and contains values from the year 1900, which cannot be real (the author says the cleaned version fixes them).

**`access_rank` is empty** in all rows, so we drop it.

**Users with an empty profile.** `join_date`, `last_online`, `stats_mean_score`, `stats_rewatched` and `stats_episodes` each miss exactly 129 values, and part 4 confirmed that they are the same 129 users. 127 of them own list rows (median declared list size 284), so they are real users whose profile summary could not be read.

## 3. List table

The file has about 80 million rows, so it is read in two ways:

1. **A sample of the first 1,000,000 rows, all columns as text.** This only shows the *formats* of the columns. The file is ordered by user, so the first million rows belong to a small group of users and must not be used for statistics.
2. **A full read of the columns that hold few distinct values or numbers** (one pass, a few minutes).

In [9]:
sample = pd.read_csv(paths["UserAnimeList.csv"], nrows=1_000_000, dtype=str, **na_kwargs)
print("sample shape:", sample.shape, "| columns:", list(sample.columns))
print(profile(sample).to_string())
print()
print(sample.head(5).T.to_string())
del sample

sample shape: (1000000, 11) | columns: ['username', 'anime_id', 'my_watched_episodes', 'my_start_date', 'my_finish_date', 'my_score', 'my_status', 'my_rewatching', 'my_rewatching_ep', 'my_last_updated', 'my_tags']
                      dtype  missing  missing_pct  distinct                                                                                       examples
column                                                                                                                                                    
username             object        0         0.00      3065                                                 [karthiga, RedvelvetDaisuki, Damonashu, bskai]
anime_id             object        0         0.00     14453                                                                              [21, 59, 74, 120]
my_watched_episodes  object        0         0.00       743                                                                             [586, 26, 161, 70]
my_start_da

In [10]:
if RUN_LISTS:
    list_cols = {
        "my_status": "int8", "my_score": "int8", "my_rewatching": "category",
        "my_watched_episodes": "float32", "my_rewatching_ep": "float32",
        "my_start_date": "category", "my_finish_date": "category",
    }
    lists = pd.read_csv(paths["UserAnimeList.csv"], usecols=list(list_cols), dtype=list_cols, **na_kwargs)
    print(f"rows: {len(lists):,}")

    status_names = {1: "watching", 2: "completed", 3: "on hold", 4: "dropped", 6: "plan to watch"}
    status_table = lists.my_status.value_counts().sort_index().to_frame("rows")
    status_table["meaning"] = status_table.index.map(status_names).fillna("unknown")
    status_table["share_pct"] = (status_table.rows / len(lists) * 100).round(3)
    print("\n--- my_status")
    print(status_table.to_string())

    print("\n--- my_score (0 = not rated)")
    print(lists.my_score.value_counts().sort_index().to_string())

    print("\n--- my_rewatching")
    print(lists.my_rewatching.value_counts(dropna=False).to_string())

    print("\n--- my_watched_episodes and my_rewatching_ep")
    print(lists[["my_watched_episodes", "my_rewatching_ep"]]
          .describe(percentiles=[.5, .99, .999]).round(1).to_string())

    for col in ["my_start_date", "my_finish_date"]:
        counts = lists[col].value_counts(dropna=False)
        print(f"\n--- {col}: {lists[col].nunique():,} distinct values | top 5:")
        print(counts.head(5).to_string())
        print("share of '0000-00-00':", f"{(lists[col] == '0000-00-00').mean():.1%}")
else:
    print("Skipped (RUN_LISTS = False)")

rows: 80,076,112

--- my_status
               rows        meaning  share_pct
my_status                                    
0               527        unknown      0.001
1           4057788       watching      5.067
2          50445526      completed     62.997
3           2985804        on hold      3.729
4           3194353        dropped      3.989
5                 6        unknown      0.000
6          19392104  plan to watch     24.217
33                3        unknown      0.000
55                1        unknown      0.000

--- my_score (0 = not rated)
my_score
0     33717790
1       321270
2       374744
3       619587
4      1296354
5      2905183
6      5456518
7     10379184
8     11394082
9      7897871
10     5713529

--- my_rewatching
my_rewatching
0      63288415
NaN    16741847
1         45850

--- my_watched_episodes and my_rewatching_ep
       my_watched_episodes  my_rewatching_ep
count           80076112.0        80076112.0
mean                  14.0               

### NEW — do users have "completed" rows for anime labelled Not yet aired? (added after the first run)

This is the third piece of evidence of the table above and the strongest one: nobody can finish an anime that has not aired. The cell reads the list table in chunks, keeps only the rows of the anime labelled `Not yet aired`, and counts them by `my_status`. It also prints the latest `my_last_updated` of the whole file (a Unix time in seconds), which is a second view of the crawl date.

**How to read the result.** Many rows with status 2 (completed) for an anime mean that its label is stale. Rows with status 6 (plan to watch) are normal, because people can add an anime to their plans before it airs.


In [11]:
# NEW: do users have completed rows for anime labelled Not yet aired?
if RUN_LISTS:
    nya_ids = set(anime.loc[anime["status"] == "Not yet aired", "anime_id"])
    parts, latest = [], 0
    reader = pd.read_csv(
        paths["UserAnimeList.csv"], usecols=["anime_id", "my_status", "my_last_updated"],
        dtype={"anime_id": "int32", "my_status": "int8", "my_last_updated": "float64"},
        chunksize=5_000_000, **na_kwargs)
    for chunk in reader:
        latest = max(latest, chunk["my_last_updated"].max())
        parts.append(chunk[chunk["anime_id"].isin(nya_ids)])
    rows = pd.concat(parts)

    print("latest my_last_updated in the whole file:", pd.to_datetime(latest, unit="s"))
    print("rows of the anime labelled Not yet aired:", f"{len(rows):,}",
          "| anime with at least one row:", rows["anime_id"].nunique(), "of", len(nya_ids))
    print()
    print(rows.groupby("my_status").size().rename("rows").to_string())
    completed = rows[rows["my_status"] == 2].groupby("anime_id").size().rename("completed_rows")
    print()
    print("anime with at least one completed row:", len(completed))
    top = completed.sort_values(ascending=False).head(10).reset_index().merge(
        anime[["anime_id", "title", "start"]], on="anime_id")
    print(top.to_string(index=False))


latest my_last_updated in the whole file: 2018-05-22 08:01:08
rows of the anime labelled Not yet aired: 543,024 | anime with at least one row: 336 of 336

my_status
1        81
2       866
3        42
4        16
6    542019

anime with at least one completed row: 6
 anime_id  completed_rows                                                   title                     start
    37831             859 Sword Art Online Alternative: Gun Gale Online - Refrain 2018-05-13 00:00:00+00:00
    36816               3                   Godzilla: Kessen Kidou Zoushoku Toshi 2018-05-18 00:00:00+00:00
    34443               1                                              Baki (ONA) 2018-07-09 00:00:00+00:00
    30455               1                    Kantai Collection: KanColle Zoku-hen                       NaT
    37284               1                              Tanuki to Kitsune Specials 2018-05-15 00:00:00+00:00
    37474               1                                    Houkago no Yuutousei 201

**Result (list table). The label is believable for almost all of the 336 anime.**

| `my_status` of the rows of these anime | Rows | Share of the 543,024 rows |
|---|---|---|
| 6 plan to watch | 542,019 | 99.8% |
| 2 completed | 866 | 0.16% |
| 1 watching | 81 | 0.01% |
| 3 on hold | 42 | 0.01% |
| 4 dropped | 16 | 0.003% |

- **Every one of the 336 anime has at least one row** (543,024 rows in total), and 99.8% of them are plans. This is exactly what an anime that has not aired should look like.
- **Only 6 anime have a completed row, and one anime holds almost all of them.** Anime 37831 (*Sword Art Online Alternative: Gun Gale Online - Refrain*, start date 2018-05-13, before the crawl bound) has 859 of the 866 completed rows (99.2%). For this anime the label is probably stale: it had started airing by the crawl. The other five have 1 to 3 completed rows each, which can be a single user's mistake (for example *Baki (ONA)* has 1 completed row and a start date in July 2018).
- **So at most 6 of 336 labels (1.8%) are doubtful by this test**, much fewer than the 60 of the date test. We trust the list rows more, because a completed row needs a real user action.
- **Proposal (the team decides in step 4):** keep `status` as given, leave the `Not yet aired` anime out of score questions (they have almost no real audience), and flag the 6 anime with completed rows, of which 2 look stale (part 4).
- **The latest `my_last_updated` of the whole file is 2018-05-22 08:01:08**, which gives a second view of the crawl date.


### Observations for part 3 (list table)

**Formats.** `my_last_updated` is a **Unix timestamp in seconds**, the number of seconds since 1 January 1970. For example, 1362307973 converts to 2013-03-03 10:52:53 UTC, the same moment a public notebook shows for this row after reading the data as dates. `my_tags` is free text typed by the users, and it is missing in 91.3% of the first million rows.

**Dates are mostly empty.** The placeholder `0000-00-00` fills 86.2% of `my_start_date` and 83.1% of `my_finish_date`. Some dates are partial (`2008-00-00`, `2012-08-00`), meaning the day or the month is unknown. These two columns cannot support a timeline. If we need time, `my_last_updated` is the only complete date.

**`my_rewatching` holds three states:** 0 for 63,288,415 rows, 1 for 45,850 rows (0.06%), and empty for 16,741,847 rows (20.9%). We do not know what an empty value means: it may be "not set" or "not rewatching".

**Impossible values.**
- `my_watched_episodes` has a maximum of **65,535**, which is 2^16 − 1, the largest number a 16-bit field can hold. The 99th percentile is only 153, and the 99.9th is 500. (The 99th percentile is the value below which 99% of the rows fall.)
- `my_rewatching_ep` has a maximum of **16,777,215**, which is 2^24 − 1, and its 99.9th percentile is 0. Its mean (7.4) and standard deviation (11,109.7) are driven by a tiny number of huge values.
- A value equal to a power of two minus one looks like a storage limit or a data-entry artifact, not a real count. A cleaning rule should compare these values with the number of episodes of the anime.

**`my_status` and `my_score`** match step 1: 537 rows use unexplained status codes (0, 5, 33, 55), and score 0 means not rated (33,717,790 rows).

**What this changes for us.** We do not use `my_start_date`, `my_finish_date` or `my_rewatching_ep` as they are. We cap or flag `my_watched_episodes`. We treat `my_last_updated` as the time axis for any question about time. The next step is the step 3 notebook (`kaggle/03_raw_quality_audit.ipynb`).

## 4. Follow-up checks: three questions left open by parts 1 to 3

Running parts 1 to 3 again takes a long time, so this part holds only the code needed for three open questions. It reuses `anime` and `users`, which are loaded above, and it reads the 5 GB list table **once**, in chunks, keeping only a few thousand rows.

| # | Question | Why it matters |
|---|---|---|
| 1 | Do **both usernames of each shared-`user_id` pair** (found in part 2) have rows in the list table, and are those rows the same? | If both names own rows with the same anime, one person's list would be counted twice. |
| 2 | Are the **60 early start dates** of the anime labelled `Not yet aired` (found in part 1) real airing dates, or dates made up from a partial text? | It decides whether the label "Not yet aired" can be trusted for these 60 anime. |
| 3 | Are the users with an **empty profile** always the same 129 users, and do they own list rows? | We found 129 missing values in five profile columns. We assumed they are the same users. |

**How long it takes.** The anime and user checks take seconds. The pass over the list table reads chunks of 5 million rows and takes a few minutes. `RUN_LISTS = False` (first code cell of this notebook) skips it.


### Question 3 first (fast): the users with an empty profile

**What we found in step 2.** Five columns of the user table (`join_date`, `last_online`, `stats_mean_score`, `stats_rewatched`, `stats_episodes`) each miss exactly 129 values. We assumed these are the same 129 users, and that they are users whose profile page could not be read.

**What the cell does.** It checks that the five missing masks are identical, counts how many of these users declare anime in their list sizes (`user_watching`, ...), and checks whether the four usernames of the shared-`user_id` pairs are among them. Whether they own rows in the list table is answered in the second half of this part.


In [12]:
miss_cols = ["join_date", "last_online", "stats_mean_score", "stats_rewatched", "stats_episodes"]
masks = {c: users[c].isna() for c in miss_cols}
print("missing values per column:", {c: int(m.sum()) for c, m in masks.items()})
print("the same users miss all five columns:", all(masks[c].equals(masks["join_date"]) for c in miss_cols))

no_profile = users[masks["join_date"]]
size_cols = ["user_watching", "user_completed", "user_onhold", "user_dropped", "user_plantowatch"]
declared = no_profile[size_cols].sum(axis=1)
print("users with an empty profile:", len(no_profile))
print("  declare 0 anime:", int((declared == 0).sum()), "| declare more than 0:", int((declared > 0).sum()))
print("  declared list size: median", declared.median(), "| mean", round(declared.mean(), 1), "| max", declared.max())

dup_ids = users.loc[users["user_id"].duplicated(keep=False), "user_id"].unique()
pair_rows = users[users["user_id"].isin(dup_ids)].sort_values("user_id")
pair_names = set(pair_rows["username"])
print()
print("usernames of the shared-user_id pairs and whether their profile is empty:")
for _, r in pair_rows.iterrows():
    print(f"  user_id {r.user_id}: {r.username:15s} empty profile: {r.username in set(no_profile['username'])}")


missing values per column: {'join_date': 129, 'last_online': 129, 'stats_mean_score': 129, 'stats_rewatched': 129, 'stats_episodes': 129}
the same users miss all five columns: True
users with an empty profile: 129
  declare 0 anime: 2 | declare more than 0: 127
  declared list size: median 284.0 | mean 356.1 | max 2997

usernames of the shared-user_id pairs and whether their profile is empty:
  user_id 3303323: Io_Otonashi     empty profile: True
  user_id 3303323: Edengarde       empty profile: False
  user_id 4053497: Dojin_          empty profile: True
  user_id 4053497: DXJIN           empty profile: False


**Result: exactly 129 users have an empty profile, and they are not empty accounts.** The five missing masks are identical, so the same 129 users miss `join_date`, `last_online`, `stats_mean_score`, `stats_rewatched` and `stats_episodes`. **127 of the 129 declare anime** (median declared list size 284, largest 2,997), and only 2 declare 0. So these users have long lists, but the summary of their profile could not be read (we do not know why). Two of them are `Io_Otonashi` and `Dojin_`, the empty-profile names of the two shared-`user_id` pairs.


### The one pass over the list table (questions 1 and 2 need it)

In one pass over the 80 million rows (chunks of 5 million) the cell keeps only:
- all rows of the four usernames of the shared-`user_id` pairs (question 1);
- the rows of the anime labelled `Not yet aired`, with `my_status` and `my_score` (question 2);
- which of the empty-profile usernames own at least one row (question 3).


In [13]:
def start_date(text):
    # `aired` is the text of a Python dictionary such as {'from': '2012-01-13T00:00:00+00:00', 'to': None}
    try:
        return pd.to_datetime(ast.literal_eval(text).get("from"), utc=True, errors="coerce")
    except Exception:
        return pd.NaT

anime["start"] = anime["aired"].map(start_date)
nya = anime[anime["status"] == "Not yet aired"].copy()
bound = anime.loc[anime["status"] == "Currently Airing", "start"].max()
nya["group"] = "later or missing date"
nya.loc[nya["start"] <= bound, "group"] = "early date (on or before the bound)"
print("crawl bound (latest start date among Currently Airing):", bound)
print(nya["group"].value_counts().to_string())

if RUN_LISTS:
    nya_ids = set(nya["anime_id"])
    empty_profile_names = set(no_profile["username"])
    pair_parts, nya_parts, owners = [], [], set()
    reader = pd.read_csv(
        paths["UserAnimeList.csv"], usecols=["username", "anime_id", "my_status", "my_score"],
        dtype={"anime_id": "int32", "my_status": "int8", "my_score": "int8"},
        chunksize=5_000_000, **na_kwargs)
    for chunk in reader:
        pair_parts.append(chunk[chunk["username"].isin(pair_names)])
        nya_parts.append(chunk.loc[chunk["anime_id"].isin(nya_ids), ["anime_id", "my_status", "my_score"]])
        owners |= set(chunk.loc[chunk["username"].isin(empty_profile_names), "username"].unique())
    pair_lists = pd.concat(pair_parts)
    nya_rows = pd.concat(nya_parts)
    print("rows kept: pairs", len(pair_lists), "| Not yet aired", len(nya_rows))


crawl bound (latest start date among Currently Airing): 2018-05-25 00:00:00+00:00
group
later or missing date                  276
early date (on or before the bound)     60
rows kept: pairs 2118 | Not yet aired 543024


### Question 1: do both usernames of a pair own list rows, and are the rows the same?

**What the cell does.** For each shared `user_id` it prints how many list rows each of the two usernames owns, how many anime appear in both lists, and for how many of those anime the status and the score are also equal. It also prints the list size the profile declares, to compare with the rows.

**How to read the result.**
- **One name owns all the rows and the other owns none:** the list is counted once. The second name is just an empty duplicate of the user row, and we can drop it.
- **Both names own rows with mostly the same anime, status and score:** one person's list is stored twice, so we must keep only one name.
- **Both own rows with different anime:** they are two different lists, and the shared id is only an id error.


In [14]:
if RUN_LISTS:
    for uid, group in pair_rows.groupby("user_id"):
        a, b = group["username"].tolist()
        ra, rb = pair_lists[pair_lists["username"] == a], pair_lists[pair_lists["username"] == b]
        declared_size = int(group[size_cols].sum(axis=1).iloc[0])
        print(f"user_id {uid}: declared list size {declared_size}")
        print(f"  {a}: {len(ra)} rows | {b}: {len(rb)} rows")
        both = ra.merge(rb, on="anime_id", suffixes=("_a", "_b"))
        same = (both["my_status_a"] == both["my_status_b"]) & (both["my_score_a"] == both["my_score_b"])
        print(f"  anime in both lists: {len(both)} | of them with the same status and score: {int(same.sum())}")


user_id 3303323: declared list size 522
  Io_Otonashi: 522 rows | Edengarde: 522 rows
  anime in both lists: 522 | of them with the same status and score: 522
user_id 4053497: declared list size 537
  Dojin_: 537 rows | DXJIN: 537 rows
  anime in both lists: 537 | of them with the same status and score: 537


**Result: each pair is one list stored twice.**

| `user_id` | Usernames | List rows of each name | Declared list size | Anime in both lists | Same anime with the same status and score |
|---|---|---|---|---|---|
| 3303323 | `Io_Otonashi` / `Edengarde` | 522 / 522 | 522 | 522 | 522 |
| 4053497 | `Dojin_` / `DXJIN` | 537 / 537 | 537 | 537 | 537 |

- **Both names own a complete list, and the two lists are identical.** Every anime of one list is in the other with the same status and the same score. The declared list size equals the number of rows. So one MAL account was crawled under two usernames, and each of its list rows exists twice.
- **1,059 list rows are duplicates** (522 + 537) among 80,076,112, which is 0.0013%. The key check of notebook 01 could not see them, because the key is `(username, anime_id)` and the usernames differ.
- **Why it matters.** A count by `anime_id` counts these two people twice. The effect is tiny, but a clean dataset should not hold the same list twice.
- **What we compared.** Anime, status and score, not the dates or the episode numbers. A 100% match on 1,059 rows is already strong evidence.
- **Which name to keep.** The name with the full profile (`Edengarde`, `DXJIN`) is the natural choice, but we do not know which name is the current one. The team decides in step 4.


### Question 2: are the early start dates real airing dates?

**The idea.** Of the 336 anime labelled `Not yet aired`, 60 have a start date on or before the crawl bound. If those dates were **real airing dates**, people would already be watching and finishing these anime, so they would look **different** from the other 276: fewer plan-to-watch rows, more watching and completed rows, more raters. If the dates carry no information about airing (an expected date, or a date made up from a partial text), the two groups would look **the same**.

**What the cells do.** They compare the two groups on the share of plan-to-watch rows, the number of non-plan rows per anime, the number of anime with raters, and the months of the early dates. The last cell lists the early anime with the most non-plan rows.

**How to read the result.** Similar numbers in the two groups mean the early dates say nothing about airing. Clearly different numbers mean the label is stale for the early group.


In [15]:
if RUN_LISTS:
    rows = nya_rows.merge(nya[["anime_id", "group"]], on="anime_id")
    by_status = rows.groupby(["group", "my_status"]).size().unstack(fill_value=0)
    by_status["rows"] = by_status.sum(axis=1)
    by_status["plan_to_watch_pct"] = (by_status[6] / by_status["rows"] * 100).round(2)
    print(by_status.to_string())

    non_plan = rows[rows["my_status"] != 6].groupby("anime_id").size().rename("non_plan_rows")
    nya_rows_info = nya.merge(non_plan, on="anime_id", how="left").fillna({"non_plan_rows": 0})
    summary = nya_rows_info.groupby("group").agg(
        anime=("anime_id", "size"),
        non_plan_rows_per_anime=("non_plan_rows", "mean"),
        anime_with_raters=("scored_by", lambda s: int((s > 0).sum())),
        median_members=("members", "median"))
    print()
    print(summary.round(2).to_string())

    early = nya_rows_info[nya_rows_info["group"].str.startswith("early")]
    print()
    print("months of the early start dates:")
    print(early["start"].dt.strftime("%Y-%m").value_counts().sort_index().to_string())
    print()
    print("early anime with the most non-plan rows:")
    print(early.sort_values("non_plan_rows", ascending=False)[
        ["anime_id", "title", "start", "scored_by", "members", "non_plan_rows"]].head(10).to_string(index=False))


my_status                             1    2   3   4       6    rows  plan_to_watch_pct
group                                                                                  
early date (on or before the bound)  64  864  22  12   52597   53559              98.20
later or missing date                17    2  20   4  489422  489465              99.99

                                     anime  non_plan_rows_per_anime  anime_with_raters  median_members
group                                                                                                 
early date (on or before the bound)     60                    16.03                  2          1205.0
later or missing date                  276                     0.16                 44          2172.0

months of the early start dates:
start
2018-01     1
2018-04     3
2018-05    56

early anime with the most non-plan rows:
 anime_id                                                   title                     start  scored_by  members

**Result: for 58 of the 60 early anime the label looks right, and for 2 it looks stale.**

| Group | Anime | Rows | Plan to watch | Non-plan rows per anime | Anime with raters (`scored_by` above 0) | Median `members` |
|---|---|---|---|---|---|---|
| Early date (60) | 60 | 53,559 | 98.20% | 16.03 | 2 | 1,205 |
| Later or missing date (276) | 276 | 489,465 | 99.99% | 0.16 | 44 | 2,172 |

- **On the surface the early group differs** (98.2% against 99.99% plan to watch, 16.03 against 0.16 non-plan rows per anime). But **two anime explain almost all of it**: *Sword Art Online Alternative: Gun Gale Online - Refrain* (939 non-plan rows) and *Godzilla: Kessen Kidou Zoushoku Toshi* (17). They hold 956 of the 962 non-plan rows of the early group. The other 58 anime share only 6 non-plan rows, which is 0.10 per anime, **even fewer than the later group (0.16)**. So 58 of the 60 behave like anime that have not aired.
- **Raters point the same way.** Only 2 of the 60 early anime have raters (3.3%), against 44 of the 276 others (15.9%). Anime that had really been airing would have more raters, not fewer.
- **The dates are bunched just before the crawl, and the next sections show why.** 56 of the 60 early dates fall in May 2018, 3 in April and 1 in January. For 54 of them the date is not a real start date but one made up from a partial text.
- **Verdict.** The label is believable for 58 of the 60 early anime. The 2 with many non-plan rows have real start dates (13 and 18 May 2018), see the next sections. Together with the list-table result in part 3, at most a handful of the 336 labels are doubtful.


### NEW — are the early start dates filled in? (added after the second run)

**Why this check exists.** Question 2 judges the early dates by what users do. The dates can also be tested directly. The text `aired_string` is what the page shows (`Oct, 2007 to Oct, 2008`), and `aired` holds the same information as a dictionary of two **complete** dates. When the text gives only a month or only a year, a complete date cannot be copied from it, so some part of the date has to be made up. If the early dates were made up in that way, they say nothing about airing, and the bunching in May 2018 would have a mechanical cause.

**What the cell does.**
1. For **all** anime, it sorts the text of the start into three kinds: a full date (`Apr 3, 1998`), a month and a year (`Jul, 2018`), or a year only (`2005`). For each kind it shows how many anime have a start date in `aired`, and the most frequent days of the month and months of those dates. If the dates of the partial kinds are spread like real dates, nothing is made up. If the same day numbers and the same month come back again and again, they are made up.
2. For the 60 early anime and for the 129 anime with a later date, it shows how the start is written in the text, and it lists the 60 early anime with their text and their date side by side, so that we can see a date that the text does not contain.

**How to read the result.** The early dates are **filled in** if their text is only a month or a year while `aired` has a complete date, and if the same days and the same month repeat. They are **real** if the text is a full date and matches.


In [16]:
# NEW: are the early start dates filled in? The date in `aired` is compared with the text of `aired_string`
import re

def start_kind(t):
    first = str(t).split(" to ")[0].strip()
    if re.search(r"^[A-Z][a-z]{2} \d{1,2}, \d{4}$", first): return "full date"
    if re.search(r"^[A-Z][a-z]{2}, \d{4}$", first): return "month and year"
    if re.search(r"^\d{4}$", first): return "year only"
    return "other"

anime["start_kind"] = anime["aired_string"].map(start_kind)
print("all anime: how aired_string writes the start, and the start dates that `aired` holds for each kind")
for kind in ["full date", "month and year", "year only", "other"]:
    s = anime.loc[(anime["start_kind"] == kind) & anime["start"].notna(), "start"]
    n = int((anime["start_kind"] == kind).sum())
    line = f"  {kind:15s} anime: {n:>6,} | with a start date: {len(s):>6,}"
    if len(s):
        line += f" | day of the month, top 4: {s.dt.day.value_counts().head(4).to_dict()} | month, top 3: {s.dt.month.value_counts().head(3).to_dict()}"
    print(line)

early = nya[nya["group"].str.startswith("early")].copy()
early["start_kind"] = early["aired_string"].map(start_kind)
later = nya[nya["start"].notna() & ~nya["group"].str.startswith("early")].copy()
later["start_kind"] = later["aired_string"].map(start_kind)
print()
print(f"the {len(early)} early anime: how aired_string writes the start:", early["start_kind"].value_counts().to_dict())
print("  start dates on the 9th to the 12th of a month:", int(early["start"].dt.day.between(9, 12).sum()), "of", len(early))
print("  start dates in May 2018:", int((early["start"].dt.strftime("%Y-%m") == "2018-05").sum()))
print(f"the {len(later)} anime with a later start date: how aired_string writes the start:", later["start_kind"].value_counts().to_dict())
print()
print(early.sort_values("start")[["anime_id", "title", "aired_string", "start"]].to_string(index=False, max_colwidth=36))


all anime: how aired_string writes the start, and the start dates that `aired` holds for each kind
  full date       anime: 12,560 | with a start date: 12,560 | day of the month, top 4: {25: 799, 21: 678, 5: 623, 1: 612} | month, top 3: {4: 1959, 10: 1716, 7: 1477}
  month and year  anime:    476 | with a start date:    122 | day of the month, top 4: {11: 81, 10: 15, 12: 10, 9: 10} | month, top 3: {7: 44, 10: 34, 4: 13}
  year only       anime:  1,219 | with a start date:    105 | day of the month, top 4: {11: 71, 10: 17, 12: 7, 9: 6} | month, top 3: {5: 105}
  other           anime:    223 | with a start date:      0

the 60 early anime: how aired_string writes the start: {'year only': 49, 'full date': 6, 'month and year': 5}
  start dates on the 9th to the 12th of a month: 50 of 60
  start dates in May 2018: 56
the 129 anime with a later start date: how aired_string writes the start: {'month and year': 68, 'full date': 61}

 anime_id                                title      aired_st

**Result (are the early dates filled in?): yes, for 54 of the 60.**

| How `aired_string` writes the start | All anime | With a start date in `aired` | Early anime (60) | Later anime (129) |
|---|---|---|---|---|
| A full date (`Apr 3, 1998`) | 12,560 | 12,560 | 6 | 61 |
| A month and a year (`Jul, 2018`) | 476 | 122 | 5 | 68 |
| A year only (`2005`) | 1,219 | 105 | 49 | 0 |
| Other (`Not available`) | 223 | 0 | 0 | 0 |

- **Dates built from a partial text do not behave like real dates.** Full dates are spread over the month (the 25th 799 times, the 21st 678, the 5th 623, the 1st 612). Dates built from a partial text are not: **the same four days come back** (the 9th to the 12th for 116 of the 122 month-and-year dates and 101 of the 105 year-only dates), and for a year-only text **the month is May in all 105 cases**. The year is 2018 for 106 of the 122 and for 85 of the 105 (`Oct, 2007` gives a date in 2018). So these 227 dates are made up. The pattern (May, the 9th to the 12th, 2018) fits the date on which the anime table was read; this is an inference, we cannot see the crawler.
- **The other partial starts have no date.** 354 month-and-year texts and 1,114 year-only texts (476 - 122 and 1,219 - 105) have an empty `aired`. We did not find what decides between an empty and a made-up date.
- **54 of the 60 early dates are made up.** 49 are written as a year only (`2018 to ?`, `2019 to ?`) and 5 as a month and a year, and 50 of the 60 fall on the 9th to the 12th. Often the text places the anime in the future: `Revisions` is written `Jan, 2019 to ?` and stored as `2018-01-11`, `Carol & Tuesday` and `Senki Zesshou Symphogear XV` are written `Apr, 2019 to ?` and stored as `2018-04-10`, `Chihayafuru 3` is written `2019 to ?` and stored as `2018-05-10`. The stored date lies before the bound, so the made-up date alone creates the signal "the label looks stale".
- **Only 6 early dates are real** (a full date in the text): `Sword Art Online Alternative: Gun Gale Online - Refrain` (13 May 2018), `Tanuki to Kitsune Specials` (15 May), `Godzilla: Kessen Kidou Zoushoku Toshi` (18 May), `Mo Ri Shu Guang` (20 May), `Houkago Midnighters: Short Movies` (23 May) and `Houkago no Yuutousei` (25 May). The first and the third are the two anime with many non-plan rows. **Hypothesis:** if the anime table was read around 9 to 12 May (the days of the made-up dates) and the list table on 22 May (its latest `my_last_updated`), an anime that started on 13 or 18 May was still "not yet aired" in the anime table and already watched in the list table. Then their label is not a mistake: it describes an earlier moment than the list rows. Not tested: three anime (`Venus Blood: Brave`, `Meshinuma.`, `Neko Kikaku`) carry the day 25 instead, so the anime table may not have been read on a single day.
- **The 129 later dates** are 61 full dates and 68 written as a month and a year. For the 68 the month is the written one, but the day (and often the year) is made up.
- **Consequence.** The test "start date on or before the bound" is meaningful only for a date written in full in the text. Among the 336 anime, 67 have a real date (6 early and 61 later), 122 have a made-up date (54 and 68) and 147 have none. For step 4, read the start from `aired_string` and use `aired` only when the text is a full date. Not checked: whether the start date that defines the bound itself (2018-05-25, the latest among the `Currently Airing` anime) is a real date.
- **Verdict.** The label `Not yet aired` is believable for 58 of the 60 early anime, and the start dates of 54 of them say nothing about airing. The 2 labels that look different belong to anime that started after the likely date on which the anime table was read. At most a handful of the 336 labels are doubtful.


### Question 3 (second half): do the empty-profile users own list rows?


In [17]:
if RUN_LISTS:
    print("users with an empty profile:", len(empty_profile_names))
    print("  of them with at least one list row:", len(owners))
    print("  of them without any list row:", len(empty_profile_names - owners))
    declared_by_name = no_profile.set_index("username")[size_cols].sum(axis=1)
    no_rows = declared_by_name[list(empty_profile_names - owners)]
    print("  of those without rows, declaring more than 0 anime:", int((no_rows > 0).sum()))


users with an empty profile: 129
  of them with at least one list row: 127
  of them without any list row: 2
  of those without rows, declaring more than 0 anime: 0


**Result: the empty-profile users own list rows, so nothing is lost for them.** 127 of the 129 own at least one row. The 2 without rows declare 0 anime, so their list is really empty. None of the 8 users who declare anime but have no rows (notebook 01, 2.2 d) is an empty-profile user.


### Observations for part 4

**1. The 129 users with an empty profile are real users with lists.** The same 129 users miss all five profile columns. 127 of them own list rows, with a median declared list size of 284. We must not drop them as "empty accounts". What we lose is their profile summary (`stats_*`, `join_date`, `last_online`), so any analysis that uses those columns covers only the other 302,546 users.

**2. Two accounts are stored twice, with the same list.** `Io_Otonashi` and `Edengarde` share `user_id` 3303323, and `Dojin_` and `DXJIN` share `user_id` 4053497. In each pair both names own the identical list (522 and 537 rows, same anime, status and score), which makes **1,059 duplicated list rows**. Proposal for step 4: keep the name with the full profile and drop the other name from both tables (2 user rows and 1,059 list rows).

**3. The label "Not yet aired" is believable for almost all of the 336 anime.** For the 60 anime with an early start date, 54 have a date that was made up from a partial text (a year, or a month and a year; the days 9 to 12 and the month May come back again and again) and says nothing about airing. 58 of the 60 look like the other unaired anime in the list table (almost only plan-to-watch rows, few raters). The 2 that look different, *Sword Art Online Alternative: Gun Gale Online - Refrain* and *Godzilla: Kessen Kidou Zoushoku Toshi*, have real start dates (13 and 18 May 2018), after the days 9 to 12 May that the made-up dates point to. This fits an anime table read in the second week of May and a list table read later (hypothesis). Proposal for step 4: keep `status` as given, leave the `Not yet aired` anime out of score questions, flag the 6 anime with completed rows, and read the start of an anime from `aired_string`, using `aired` only when the text is a full date.

**What this changes in step 2.** The dictionary rows of `user_id` and `status` now cite these results, and the open question about the 129 users is closed.
